In [3]:
%cd PRO507

[Errno 2] No such file or directory: 'PRO507'
/workspace/PRO507


/usr/local/lib/python3.10/site-packages/IPython/core/magics/osm.py:393: UserWarning: This is now an optional IPython functionality, using bookmarks requires you to install the `pickleshare` library.
  bkms = self.shell.db.get('bookmarks', {})


In [4]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField
from pyspark.sql.types import StringType, IntegerType, FloatType,BooleanType,DateType,TimestampType
from pyspark.sql import functions as fun
from pyspark.sql import Window
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import ParamGridBuilder,CrossValidator

import pandas as pd 

spark = ( SparkSession.builder.appName("PRO507").master("spark://spark-master:7077").getOrCreate())



sc = spark.sparkContext

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/05 09:35:42 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [6]:
als =  ALS(
userCol="user_id",
itemCol="venue_id",
ratingCol="rating",
coldStartStrategy="drop",
)

grid_paramas = (ParamGridBuilder()
                .addGrid(als.rank,[5,10,15])
                .addGrid(als.regParam,[0.01,0.1])
                .addGrid(als.maxIter,[10])
                .build())

evaluador = RegressionEvaluator(
    metricName="rmse",
    labelCol="rating",
    predictionCol="prediction"
)

validador_cruzado = CrossValidator(
    estimator=als,
    estimatorParamMaps=grid_paramas,
    evaluator=evaluador,
    numFolds=3
)

modelo_optimizado = validador_cruzado.fit(train)

26/03/05 09:35:58 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [ ]:
# combinaciones = modelo_optimizado.getEstimatorParamMaps()

# notas_rsme = modelo_optimizado.avgMetrics

# ranking = sorted(zip(notas_rsme,combinaciones),key=lambda x:x[0])

# for nota , parametros in ranking:
#     print(f"rsme obtenido: {nota}")

#     for param, valor in parametros.items():
#         print(f"{param.name}: {valor}")

best_model = modelo_optimizado.bestModel

# 2. Obtener los valores de los parámetros ganadores
v_rank = best_model.rank
v_reg  = best_model._java_obj.parent().getRegParam()
v_iter = best_model._java_obj.parent().getMaxIter()

# 3. Imprimir por pantalla de forma limpia

print(f"Rank: {v_rank}")
print(f"RegParam: {v_reg}")
print(f"MaxIter: {v_iter}")


------------------------------
MEJORES HIPERPARÁMETROS:
Rank: 15
RegParam: 0.1
MaxIter: 10
------------------------------


In [ ]:
from pyspark.sql.functions import explode

user_to_recomend = spark.createDataFrame([1],["user_id"])

user_recs = best_model.

